# Trợ lý ôn tập có trích dẫn nguồn

**Sổ Nghiên Cứu AI, dự án mẫu 9/10.** Truy xuất đoạn tài liệu liên quan trước khi trả lời, đo khả năng tìm đúng nguồn và tỉ lệ trả lời bịa.

**Câu hỏi nghiên cứu:** Truy xuất tăng cường (RAG) có giảm tỉ lệ trả lời sai sự thật của mô hình ngôn ngữ trên câu hỏi Sinh học không?

**Giả thuyết:** Hệ thống RAG có tỉ lệ trả lời sai thấp hơn mô hình không truy xuất và trích đúng đoạn nguồn trong phần lớn câu trả lời.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Tài liệu mẫu gồm 12 đoạn kiến thức Sinh học ngắn do Sổ Nghiên Cứu AI tự viết để luyện tập. Khi làm thật, dùng tài liệu do giáo viên soạn và cho phép.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
%pip install -q sentence-transformers

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
CORPUS = [('Quang hợp', 'Quang hợp ở thực vật diễn ra chủ yếu ở lục lạp của tế bào lá. Cây dùng năng lượng ánh sáng mặt trời để tổng hợp chất hữu cơ từ CO2 và nước, đồng thời giải phóng khí O2.'), ('Hai pha quang hợp', 'Quang hợp gồm pha sáng và pha tối. Pha sáng diễn ra ở màng tilacoit, tạo ATP và NADPH, khí O2 được giải phóng từ quá trình quang phân li nước. Pha tối diễn ra ở chất nền lục lạp, cố định CO2 qua chu trình Calvin.'), ('Thực vật C4 và CAM', 'Thực vật C4 như ngô, mía cố định CO2 hai lần nên hầu như không có hô hấp sáng và năng suất cao. Thực vật CAM như dứa, xương rồng mở khí khổng vào ban đêm để tiết kiệm nước.'), ('Hô hấp tế bào', 'Hô hấp ở thực vật phân giải chất hữu cơ thành CO2 và nước, giải phóng năng lượng dưới dạng ATP. Hô hấp hiếu khí gồm đường phân ở tế bào chất, chu trình Krebs ở chất nền ti thể và chuỗi truyền electron ở màng trong ti thể.'), ('Trao đổi nước', 'Rễ cây hút nước chủ yếu nhờ các tế bào lông hút. Nước thoát ra khỏi lá chủ yếu qua khí khổng. Thoát hơi nước tạo động lực kéo nước lên và giúp hạ nhiệt độ của lá.'), ('Hô hấp ở động vật', 'Giun đất trao đổi khí qua bề mặt cơ thể, côn trùng dùng hệ thống ống khí, cá hô hấp bằng mang. Chim hô hấp bằng phổi và hệ thống túi khí nên khí qua phổi theo một chiều, trao đổi khí rất hiệu quả.'), ('Hệ tuần hoàn', 'Hệ tuần hoàn hở có ở đa số thân mềm và chân khớp. Hệ tuần hoàn kín có ở mực, bạch tuộc, giun đốt và động vật có xương sống. Cá có tuần hoàn đơn; lưỡng cư, bò sát, chim và thú có tuần hoàn kép.'), ('Tính tự động của tim', 'Tim có khả năng tự co dãn nhịp nhàng nhờ hệ dẫn truyền tim gồm nút xoang nhĩ, nút nhĩ thất, bó His và mạng Purkinje. Nút xoang nhĩ tự phát xung điện và được xem là bộ phận phát nhịp của tim.'), ('Cảm ứng ở thực vật', 'Hướng động là vận động sinh trưởng theo một hướng xác định của kích thích, ví dụ hướng sáng, hướng trọng lực. Ứng động không theo hướng của kích thích; hiện tượng cụp lá ở cây trinh nữ khi bị chạm là ứng động do thay đổi sức trương nước.'), ('Điện thế nghỉ', 'Điện thế nghỉ là sự chênh lệch điện thế giữa hai bên màng tế bào khi tế bào không bị kích thích, trong đó mặt trong màng tích điện âm so với mặt ngoài. Ở nơron, điện thế nghỉ vào khoảng âm 70 mV.'), ('Hormone thực vật', 'Auxin kích thích sự dãn dài của tế bào. Gibberellin kích thích nảy mầm và kéo dài thân. Cytokinin kích thích phân chia tế bào. Abscisic acid gây đóng khí khổng và ngủ nghỉ của hạt. Ethylene thúc quả chín.'), ('Tiêu hóa ở người', 'Ở người, tinh bột bắt đầu được tiêu hóa hóa học ở khoang miệng nhờ enzyme amylase trong nước bọt. Phần lớn chất dinh dưỡng được hấp thụ ở ruột non nhờ các nếp gấp, lông ruột và lông cực nhỏ làm tăng diện tích bề mặt.')]
QS = [('Pha sáng của quang hợp diễn ra ở đâu?', 1), ('Vì sao cây ngô có năng suất quang hợp cao?', 2), ('Cây dứa mở khí khổng vào lúc nào?', 2), ('Chu trình Krebs xảy ra ở vị trí nào trong tế bào?', 3), ('Bộ phận nào phát nhịp cho tim?', 7), ('Cá có kiểu tuần hoàn gì?', 6), ('Hiện tượng cây trinh nữ cụp lá thuộc loại cảm ứng nào?', 8), ('Điện thế nghỉ của nơron khoảng bao nhiêu?', 9), ('Hormone nào làm quả chín?', 10), ('Nước thoát khỏi lá chủ yếu qua đâu?', 4), ('Chất dinh dưỡng được hấp thụ chủ yếu ở đâu?', 11), ('Vì sao chim trao đổi khí hiệu quả?', 5), ('Phương trình E = mc² do ai đề xuất?', None), ('Thủ đô của Pháp là gì?', None)]
if os.path.exists("tai_lieu.csv"): CORPUS = list(pd.read_csv("tai_lieu.csv")[["tieu_de", "noi_dung"]].itertuples(index=False, name=None))
if os.path.exists("cau_hoi.csv"):
    q = pd.read_csv("cau_hoi.csv"); QS = [(r.cau_hoi, None if pd.isna(r.doan_dung) else int(r.doan_dung)) for r in q.itertuples()]
print(len(CORPUS), "đoạn tài liệu |", len(QS), "câu hỏi, trong đó", sum(g is None for _, g in QS), "câu KHÔNG có trong tài liệu")

## 1. Hai cách chia đoạn và hai cách tìm kiếm
Chia đoạn: giữ nguyên mỗi đoạn, hoặc tách thành từng câu. Tìm kiếm: TF-IDF (đối chứng, dựa trên từ trùng) và vector ngữ nghĩa đa ngôn ngữ.

In [ ]:
import re
def chia(kieu):
    if kieu == "đoạn": return [(t + ". " + c, i) for i, (t, c) in enumerate(CORPUS)]
    return [(t + ". " + s.strip(), i) for i, (t, c) in enumerate(CORPUS) for s in re.split(r"(?<=\.)\s+", c) if s.strip()]
try:
    from sentence_transformers import SentenceTransformer
    emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"); CO_VECTOR = True
except Exception as e:
    print("Không tải được mô hình vector (", type(e).__name__, "), chỉ chạy TF-IDF."); CO_VECTOR = False
def tim(kieu_chia, kieu_tim):
    doan = chia(kieu_chia); van = [d for d, _ in doan]; cau = [q for q, _ in QS]
    if kieu_tim == "TF-IDF":
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit(van + cau); S = cosine_similarity(v.transform(cau), v.transform(van))
    else:
        S = cosine_similarity(emb.encode(cau, normalize_embeddings=True), emb.encode(van, normalize_embeddings=True))
    return doan, S
KQ = {}
for kc in ["đoạn", "câu"]:
    for kt in (["TF-IDF", "Vector"] if CO_VECTOR else ["TF-IDF"]):
        doan, S = tim(kc, kt); h1 = h3 = n = 0
        for qi, (q, g) in enumerate(QS):
            if g is None: continue
            top = [doan[j][1] for j in np.argsort(-S[qi])]; n += 1; h1 += top[0] == g; h3 += g in top[:3]
        KQ[f"{kt}, chia theo {kc}"] = {"Tìm đúng ở vị trí 1": h1 / n, "Đúng trong 3 kết quả đầu": h3 / n}
print(pd.DataFrame(KQ).T.round(2))

## 2. Khi nào nên trả lời “không có trong tài liệu”?
Xem điểm tương đồng cao nhất của câu hỏi có đáp án và câu hỏi không có đáp án. Nếu hai nhóm tách biệt, ta đặt được ngưỡng từ chối.

In [ ]:
doan, S = tim("đoạn", "Vector" if CO_VECTOR else "TF-IDF")
co = [S[i].max() for i, (_, g) in enumerate(QS) if g is not None]; khong = [S[i].max() for i, (_, g) in enumerate(QS) if g is None]
plt.figure(figsize=(6, 2.5)); plt.scatter(co, [1] * len(co), label="có trong tài liệu"); plt.scatter(khong, [0] * len(khong), c="r", label="không có")
plt.yticks([0, 1], ["không có", "có"]); plt.xlabel("điểm tương đồng cao nhất"); plt.legend(fontsize=7); plt.show()
NGUONG = (min(co) + max(khong)) / 2 if min(co) > max(khong) else np.percentile(co, 10)
print("Ngưỡng từ chối gợi ý:", round(NGUONG, 3))

## 3. Sinh câu trả lời (tùy chọn, nên bật GPU)
So sánh mô hình ngôn ngữ **có** và **không có** tài liệu. Kết quả được xuất ra file để giáo viên **chấm mù** (không biết câu nào từ hệ thống nào).

In [ ]:
CHAY_LLM = False                          # đổi thành True để chạy
TEN_LLM = "Qwen/Qwen2.5-0.5B-Instruct"     # mô hình mở, nhỏ, hiểu tiếng Việt; có thể đổi mô hình khác
if CHAY_LLM:
    from transformers import pipeline
    gen = pipeline("text-generation", model=TEN_LLM, device_map="auto")
    def hoi(q, ngu_canh=None):
        he = "Em là trợ lý ôn tập Sinh học. Trả lời ngắn gọn bằng tiếng Việt."
        if ngu_canh: he += " CHỈ dùng thông tin trong tài liệu dưới đây. Nếu tài liệu không có, trả lời đúng câu: Không có trong tài liệu.\n\nTài liệu:\n" + ngu_canh
        out = gen([{"role": "system", "content": he}, {"role": "user", "content": q}], max_new_tokens=120, do_sample=False)
        return out[0]["generated_text"][-1]["content"].strip()
    dong = []
    for qi, (q, g) in enumerate(QS):
        top = np.argsort(-S[qi])[:2]; nc = "\n".join(doan[j][0] for j in top) if S[qi].max() >= NGUONG else ""
        dong += [{"cau_hoi": q, "he_thong": "khong_RAG", "tra_loi": hoi(q)},
                 {"cau_hoi": q, "he_thong": "RAG", "tra_loi": hoi(q, nc) if nc else "Không có trong tài liệu.", "nguon": "; ".join(CORPUS[doan[j][1]][0] for j in top) if nc else ""}]
    B = pd.DataFrame(dong).sample(frac=1, random_state=SEED).reset_index(drop=True); B["ma"] = range(1, len(B) + 1)
    B[["ma", "cau_hoi", "tra_loi"]].assign(diem_dung_0_1="", co_bia_0_1="").to_csv("phieu_cham_mu.csv", index=False)
    B[["ma", "he_thong", "nguon"]].to_csv("khoa_dap_an.csv", index=False)
    print("Đã tạo phieu_cham_mu.csv (đưa giáo viên chấm) và khoa_dap_an.csv (em giữ lại).")
else:
    print("Bỏ qua phần sinh câu trả lời. Đổi CHAY_LLM = True khi sẵn sàng.")

## Việc của em
1. Xin phép giáo viên dùng tài liệu ôn tập của một chương, chuyển thành `tai_lieu.csv` (cột `tieu_de, noi_dung`).
2. Cùng giáo viên viết 150 câu hỏi kèm đáp án vào `cau_hoi.csv` (cột `cau_hoi, doan_dung`, để trống `doan_dung` nếu câu hỏi nằm ngoài tài liệu).
3. Sau khi giáo viên chấm `phieu_cham_mu.csv`, ghép với `khoa_dap_an.csv` để tính tỉ lệ đúng và tỉ lệ bịa của từng hệ thống.